# 💰 Valuation Sandbox

Interactive DCF modeling, Comps analysis, and scenario testing.

**Usage:** Enter a ticker, adjust parameters, explore fair value ranges.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Any

from src.data_engine import (
    get_cash_flow, get_income_statement, get_balance_sheet,
    get_key_metrics, get_quote, get_profile, get_estimates_consensus,
    get_peer_metrics,
)

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

print('✅ Valuation Sandbox Ready.')

## 1. Select Company & Auto-Load Data

In [ ]:
TICKER = 'NVDA'  # ← Change to analyze any company

# Fetch all needed data
cf = get_cash_flow(TICKER, period='annual', limit=3)
income = get_income_statement(TICKER, period='annual', limit=5)
balance = get_balance_sheet(TICKER, period='annual', limit=3)
metrics = get_key_metrics(TICKER, limit=1)
quote = get_quote(TICKER)
estimates = get_estimates_consensus(TICKER)

# Display what we have
current_price = quote.iloc[0]['close'] if not quote.empty and 'close' in quote.columns else None
print(f"{TICKER} — Current Price: ${current_price:.2f}" if current_price else f"{TICKER}")
print(f"  Income stmts: {len(income)} periods")
print(f"  Cash flow: {len(cf)} periods")
print(f"  Balance sheet: {len(balance)} periods")
print(f"  Metrics: {len(metrics)} periods")

## 2. DCF Model with Auto-Populated Parameters

In [ ]:
@dataclass
class DCFParams:
    free_cash_flow: float
    growth_rate: float
    terminal_growth: float = 0.03
    discount_rate: float = 0.10
    shares_outstanding: float = 1.0
    net_debt: float = 0.0
    years: int = 5

def run_dcf(params: DCFParams) -> dict:
    annual_fcfs = []
    fcf = params.free_cash_flow
    for _ in range(params.years):
        fcf *= (1 + params.growth_rate)
        annual_fcfs.append(fcf)
    terminal_fcf = annual_fcfs[-1] * (1 + params.terminal_growth)
    terminal_value = terminal_fcf / (params.discount_rate - params.terminal_growth)
    pv_fcfs = sum(fcf / (1 + params.discount_rate) ** (i + 1) for i, fcf in enumerate(annual_fcfs))
    pv_terminal = terminal_value / (1 + params.discount_rate) ** params.years
    ev = pv_fcfs + pv_terminal
    market_cap = ev - params.net_debt
    fair_value = market_cap / params.shares_outstanding
    return {
        'fair_value': fair_value, 'enterprise_value': ev,
        'annual_fcfs': annual_fcfs, 'terminal_value': terminal_value,
        'pv_fcfs': pv_fcfs, 'pv_terminal': pv_terminal,
    }

# Auto-populate parameters
fcf_ttm = 10000  # Default
if not cf.empty:
    for col in ['free_cash_flow']:
        if col in cf.columns:
            fcf_ttm = float(cf.iloc[0][col] / 1_000_000)
            break

growth = 0.15
if not income.empty and 'total_revenue' in income.columns and len(income) >= 2:
    revs = [float(r) for r in income['total_revenue'].dropna().values if r]
    if len(revs) >= 2 and revs[-1] > 0:
        growth = max(0.05, min(0.50, (revs[0] / revs[-1]) ** (1/(len(revs)-1)) - 1))

shares = 1000
if not metrics.empty:
    for col in ['shares_outstanding', 'weighted_average_shares']:
        if col in metrics.columns:
            val = metrics.iloc[0][col]
            if val and pd.notna(val):
                shares = float(val) / 1_000_000
                break

net_debt = 0
if not balance.empty:
    debt = float(balance.iloc[0].get('total_debt', 0) or 0) / 1_000_000
    cash = float(balance.iloc[0].get('cash_and_equivalents', 0) or 0) / 1_000_000
    net_debt = debt - cash

params = DCFParams(
    free_cash_flow=fcf_ttm,
    growth_rate=growth,
    shares_outstanding=shares,
    net_debt=net_debt,
)

print(f'### DCF Parameters for {TICKER}')
print(f'  FCF (TTM):        ${params.free_cash_flow:,.0f}M')
print(f'  Growth Rate:      {params.growth_rate:.1%}')
print(f'  Discount Rate:    {params.discount_rate:.1%}')
print(f'  Terminal Growth:  {params.terminal_growth:.1%}')
print(f'  Shares Out:       {params.shares_outstanding:,.0f}M')
print(f'  Net Debt:         ${params.net_debt:,.0f}M')

result = run_dcf(params)
upside = (result['fair_value'] / current_price - 1) * 100 if current_price else 0
print(f'\n  🔹 Fair Value: ${result["fair_value"]:.2f}')
print(f'  🔹 Current Price: ${current_price:.2f}' if current_price else '  🔹 Current: N/A')
print(f'  🔹 Upside/Downside: {upside:+.1f}%')

## 3. Sensitivity Analysis Heatmap

In [ ]:
wacc_range = np.arange(max(params.terminal_growth + 0.01, 0.06), 0.16, 0.01)
growth_range = np.arange(max(0.05, params.growth_rate - 0.15), params.growth_rate + 0.16, 0.02)

matrix = np.zeros((len(wacc_range), len(growth_range)))
for i, wacc in enumerate(wacc_range):
    for j, g in enumerate(growth_range):
        p = DCFParams(
            free_cash_flow=params.free_cash_flow,
            growth_rate=g,
            discount_rate=wacc,
            shares_outstanding=params.shares_outstanding,
            net_debt=params.net_debt,
        )
        r = run_dcf(p)
        matrix[i, j] = r['fair_value']

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto')

ax.set_xticks(range(len(growth_range)))
ax.set_xticklabels([f'{g:.0%}' for g in growth_range], rotation=45)
ax.set_yticks(range(len(wacc_range)))
ax.set_yticklabels([f'{w:.0%}' for w in wacc_range])
ax.set_xlabel('Growth Rate')
ax.set_ylabel('Discount Rate (WACC)')
ax.set_title(f'{TICKER} — DCF Sensitivity: Fair Value / Share')

# Annotate cells with values
for i in range(len(wacc_range)):
    for j in range(len(growth_range)):
        val = matrix[i, j]
        color = 'white' if val < matrix.mean() else 'black'
        ax.text(j, i, f'${val:.0f}', ha='center', va='center', fontsize=8, color=color, fontweight='bold')

plt.colorbar(im, ax=ax, label='Fair Value ($)', format='${x:,.0f}')
plt.tight_layout()
plt.show()

## 4. Scenario Analysis (Bull / Base / Bear)

In [ ]:
scenarios = {
    '🐂 Bull':    {'growth': params.growth_rate + 0.10, 'wacc': params.discount_rate - 0.02},
    '📊 Base':    {'growth': params.growth_rate,        'wacc': params.discount_rate},
    '🐻 Bear':    {'growth': max(0.02, params.growth_rate - 0.10), 'wacc': params.discount_rate + 0.02},
    '💀 Worst':   {'growth': 0.02, 'wacc': params.discount_rate + 0.04},
}

print(f"{'Scenario':<15} {'Growth':>8} {'WACC':>8} {'Fair Value':>12} {'Upside':>10}")
print('-' * 55)

results_by_scenario = {}
for name, s in scenarios.items():
    p = DCFParams(
        free_cash_flow=params.free_cash_flow,
        growth_rate=s['growth'],
        discount_rate=s['wacc'],
        shares_outstanding=params.shares_outstanding,
        net_debt=params.net_debt,
    )
    r = run_dcf(p)
    results_by_scenario[name] = r
    upside = (r['fair_value'] / current_price - 1) * 100 if current_price else 0
    print(f"{name:<15} {s['growth']:>8.1%} {s['wacc']:>8.1%} ${r['fair_value']:>11,.2f} {upside:>+9.1f}%")

## 5. Comps Quick Analysis

In [ ]:
# Define peer groups
PEER_GROUPS = {
    'NVDA': ['AMD', 'INTC', 'AVGO', 'QCOM', 'MRVL'],
    'AVGO': ['NVDA', 'AMD', 'MRVL', 'QCOM', 'INTC'],
    'ORCL': ['MSFT', 'CRM', 'ADBE', 'SAP', 'IBM'],
}

peers = PEER_GROUPS.get(TICKER.upper(), ['AMD', 'INTC'])
all_tickers = [TICKER] + peers

comp_df = get_peer_metrics(all_tickers)

if not comp_df.empty:
    target = comp_df[comp_df['ticker'] == TICKER.upper()]
    peer_df = comp_df[comp_df['ticker'] != TICKER.upper()]
    
    median_pe = peer_df['pe_ratio'].dropna().median()
    median_ev = peer_df['ev_to_ebitda'].dropna().median()
    
    print(f'Peer Group: {", ".join(peers)}')
    print(f'Median P/E: {median_pe:.1f}x')
    print(f'Median EV/EBITDA: {median_ev:.1f}x' if 'ev_to_ebitda' in peer_df.columns else '')
    print()
    display(comp_df[['ticker', 'pe_ratio', 'ev_to_ebitda', 'revenue_growth', 'gross_margin', 'roe']].round(1))
else:
    print('⚠️  No comps data.')

## 6. Monte Carlo DCF (Optional)

Simple Monte Carlo simulation: vary growth and WACC within normal distributions.

In [ ]:
N_SIMULATIONS = 1000

np.random.seed(42)
sim_growth = np.random.normal(params.growth_rate, 0.08, N_SIMULATIONS)
sim_growth = np.clip(sim_growth, 0.01, 0.60)
sim_wacc = np.random.normal(params.discount_rate, 0.02, N_SIMULATIONS)
sim_wacc = np.clip(sim_wacc, params.terminal_growth + 0.01, 0.20)

sim_prices = []
for g, w in zip(sim_growth, sim_wacc):
    p = DCFParams(
        free_cash_flow=params.free_cash_flow,
        growth_rate=g, discount_rate=w,
        shares_outstanding=params.shares_outstanding,
        net_debt=params.net_debt,
    )
    sim_prices.append(run_dcf(p)['fair_value'])

sim_prices = np.array(sim_prices)
p5, p25, p50, p75, p95 = np.percentile(sim_prices, [5, 25, 50, 75, 95])

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(sim_prices, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(x=current_price, color='red', linestyle='--', linewidth=2, label=f'Current: ${current_price:.2f}')
ax.axvline(x=p50, color='green', linestyle='-', linewidth=2, label=f'Median FV: ${p50:.2f}')
ax.axvspan(p25, p75, alpha=0.2, color='gray', label='25th-75th Percentile')
ax.set_title(f'{TICKER} — Monte Carlo DCF ({N_SIMULATIONS} simulations)')
ax.set_xlabel('Fair Value / Share ($)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Fair Value Distribution:')
print(f'  P5:   ${p5:.2f}')
print(f'  P25:  ${p25:.2f}')
print(f'  P50:  ${p50:.2f}')
print(f'  P75:  ${p75:.2f}')
print(f'  P95:  ${p95:.2f}')
print(f'  Current Price: ${current_price:.2f}' if current_price else '  Current: N/A')
if current_price:
    prob_above = (sim_prices > current_price).mean() * 100
    print(f'  Probability FV > Current: {prob_above:.1f}%')

---
*Generated by AI Investment System — Valuation Sandbox*